# Advanced Logit Decomposition

Fine-grained logit decomposition: per-component contributions,
interaction terms, direct vs indirect paths, logit lens decomposition,
and token-specific attribution.

## Why This Matters

Composition logit decomposition advanced measures how components interact across layers to implement complex computations. Understanding composition — how one head's output feeds into another's input — is essential for tracing multi-step algorithms in transformer circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.logit_decomposition_advanced import (
    per_component_logit_contribution,
    logit_interaction_terms,
    direct_vs_indirect_logit,
    logit_lens_decomposition,
    token_specific_attribution,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])
print("Model ready")

## Per-Component Logit Contribution

In [ ]:
result = per_component_logit_contribution(model, tokens)
print(f"Target token: {result['target_token']}")
print(f"Embed: {result['embed_contribution']:.4f}")
for l in range(cfg.n_layers):
    attn = float(result['attn_contributions'][l])
    mlp = float(result['mlp_contributions'][l])
    print(f"  Layer {l}: attn={attn:.4f}, mlp={mlp:.4f}")
print(f"Bias: {result['bias_contribution']:.4f}")
print(f"Total: {result['total_logit']:.4f}")

## Logit Interaction Terms

In [ ]:
result = logit_interaction_terms(model, tokens)
print(f"Single effects: {[f'{float(e):.4f}' for e in result['single_effects']]}")
print(f"\nSynergistic pairs: {len(result['synergistic_pairs'])}")
for c1, c2, v in result['synergistic_pairs'][:3]:
    print(f"  {c1} + {c2}: {v:.6f}")
print(f"Canceling pairs: {len(result['canceling_pairs'])}")

## Direct vs Indirect Contributions

In [ ]:
result = direct_vs_indirect_logit(model, tokens)
print(f"Direct fraction: {result['direct_fraction']:.1%}")
print(f"Indirect fraction: {result['indirect_fraction']:.1%}")
for l in range(cfg.n_layers):
    d_attn = float(result['direct_contributions'][2*l])
    d_mlp = float(result['direct_contributions'][2*l+1])
    i_attn = float(result['indirect_contributions'][2*l])
    i_mlp = float(result['indirect_contributions'][2*l+1])
    print(f"  L{l}: direct(attn={d_attn:.4f}, mlp={d_mlp:.4f}), indirect(attn={i_attn:.4f}, mlp={i_mlp:.4f})")

## Logit Lens Decomposition

In [ ]:
result = logit_lens_decomposition(model, tokens)
for l in range(cfg.n_layers):
    logit = float(result['logit_trajectory'][l])
    delta = float(result['delta_per_layer'][l])
    attn = float(result['attn_delta'][l])
    mlp = float(result['mlp_delta'][l])
    print(f"  Layer {l}: logit={logit:.4f}, delta={delta:+.4f} (attn={attn:+.4f}, mlp={mlp:+.4f})")

## Token-Specific Attribution

In [ ]:
result = token_specific_attribution(model, tokens)
for target, info in result['per_target'].items():
    print(f"\nTarget token {target} (base logit={info['base_logit']:.4f}):")
    for pos, effect in info['top_positions'][:3]:
        print(f"  Source pos {pos}: effect={effect:.4f}")